# Google API Core - Core Library for Google APIs

Google API Core contains the core functionality for Google API client libraries, including authentication, retries, timeouts, and exception handling.

## Example 1: Basic Client Configuration and Options

In [ ]:
# Example 1: ClientOptions, the configuration object every generated Google API client accepts
from google.api_core import client_options

options = client_options.ClientOptions(
    api_endpoint="example.googleapis.com",
    quota_project_id="demo-project",
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
print(f"api_endpoint={options.api_endpoint} quota_project_id={options.quota_project_id}")
assert options.api_endpoint == "example.googleapis.com"
assert options.quota_project_id == "demo-project"
assert options.scopes == ["https://www.googleapis.com/auth/cloud-platform"]
assert options.api_key is None and options.client_cert_source is None

# from_dict builds the same object from a plain mapping, which is how clients accept dict options.
from_mapping = client_options.from_dict({"api_endpoint": "other.example.com", "universe_domain": "example.com"})
assert isinstance(from_mapping, client_options.ClientOptions)
assert (from_mapping.api_endpoint, from_mapping.universe_domain) == ("other.example.com", "example.com")

# Unknown keys and conflicting certificate sources are rejected.
for bad in (
    lambda: client_options.from_dict({"not_an_option": 1}),
    lambda: client_options.ClientOptions(client_cert_source=lambda: None,
                                         client_encrypted_cert_source=lambda: None),
):
    try:
        bad()
    except ValueError as e:
        print(f"rejected: {e}")
    else:
        raise AssertionError("invalid ClientOptions were accepted")

## Example 2: Retry Configuration and Policies

In [ ]:
from google.api_core import retry
from google.api_core import exceptions
import time
import random

# Create a basic retry configuration
retry_config = retry.Retry(
    initial=1.0,        # Initial delay
    maximum=10.0,       # Maximum delay
    multiplier=2.0,     # Backoff multiplier
    deadline=60.0       # Total deadline
)

print("Retry Configuration:")
print(f"Initial delay: {retry_config._initial}")
print(f"Maximum delay: {retry_config._maximum}")
print(f"Multiplier: {retry_config._multiplier}")
print(f"Deadline: {retry_config._deadline}")

# Create a custom retry predicate
def should_retry(exception):
    """Custom retry predicate for testing"""
    return isinstance(exception, (exceptions.ServiceUnavailable, exceptions.TooManyRequests))

custom_retry = retry.Retry(
    predicate=should_retry,
    initial=0.5,
    maximum=5.0
)

# Test retry with a mock function
call_count = 0

def unreliable_function():
    global call_count
    call_count += 1
    if call_count < 3:
        raise exceptions.ServiceUnavailable("Service temporarily unavailable")
    return f"Success after {call_count} attempts"

# Reset counter and test
call_count = 0
print(f"\nTesting retry mechanism:")
try:
    result = custom_retry(unreliable_function)()
    print(f"Result: {result}")
except Exception as e:
    print(f"Failed after retries: {e}")
    raise

## Example 3: Exception Handling and Google API Errors

In [ ]:
from google.api_core import exceptions
from google.rpc import status_pb2, code_pb2

# Test different Google API exception types
print("Google API Exception Types:")

# Create various exception instances
bad_request = exceptions.BadRequest("Invalid request parameter")
not_found = exceptions.NotFound("Resource not found")
forbidden = exceptions.Forbidden("Access denied")
service_unavailable = exceptions.ServiceUnavailable("Service down")

exceptions_list = [bad_request, not_found, forbidden, service_unavailable]

for exc in exceptions_list:
    print(f"{exc.__class__.__name__}: {exc.message}")
    print(f"  - HTTP Status Code: {exc.code}")
    print(f"  - Is retriable: {exc.code in (503, 500, 502, 504)}")

# Test exception with gRPC status
print(f"\nTesting gRPC status integration:")
# Create a status proto
status_proto = status_pb2.Status(
    code=code_pb2.INVALID_ARGUMENT,
    message="Invalid field value"
)

# Create exception from status: from_grpc_status(status_code, message) takes the proto's integer
# code and message, not the proto itself, and maps the code to its exception class.
grpc_exception = exceptions.from_grpc_status(status_proto.code, status_proto.message)
print(f"gRPC status {status_proto.code} maps to {type(grpc_exception).__name__}")
print(f"Message: {grpc_exception.message}")
assert isinstance(grpc_exception, exceptions.InvalidArgument), type(grpc_exception)
assert grpc_exception.message == "Invalid field value" and grpc_exception.code == 400

# Test exception catching
print(f"\nTesting exception hierarchy:")
try:
    raise exceptions.InvalidArgument("Test invalid argument")
except exceptions.BadRequest as e:
    print(f"✓ Caught as BadRequest: {e.message}")
except Exception as e:
    print(f"Caught as general exception: {e}")
    raise

## Example 4: Path Template Processing

In [ ]:
import google.api_core.path_template as path_template

templates = {
    "project": "projects/{project}",
    "instance": "projects/{project}/instances/{instance}",
    "table": "projects/{project}/instances/{instance}/tables/{table}",
    "user": "users/{user_id}/profile/{profile_type}",
}
test_data = {
    "project": "my-test-project-123",
    "instance": "production-instance",
    "table": "user-data",
    "user_id": "12345",
    "profile_type": "public",
}
expected = {
    "project": "projects/my-test-project-123",
    "instance": "projects/my-test-project-123/instances/production-instance",
    "table": "projects/my-test-project-123/instances/production-instance/tables/user-data",
    "user": "users/12345/profile/public",
}

print("Path Template Examples:")
for name, template_str in templates.items():
    # expand() fills only the variables named in the template, so the full mapping can be passed.
    path = path_template.expand(template_str, **test_data)
    print(f"  {name}: {path}")
    assert path == expected[name], (name, path)

    # validate(tmpl, path) checks a concrete path against the template.
    assert path_template.validate(template_str, path), f"{path!r} does not validate against {template_str!r}"
    assert not path_template.validate(template_str, path + "/extra"), f"an over-long path validated against {template_str!r}"

# Positional wildcards and a missing named variable.
assert path_template.expand("projects/*/instances/*", "p", "i") == "projects/p/instances/i"
try:
    path_template.expand("projects/{project}")
except ValueError as e:
    print(f"missing variable rejected: {e}")
else:
    raise AssertionError("expand() accepted a template with an unfilled variable")

print("\nPath template testing completed!")

## Example 5: Timeout and Deadline Management

In [ ]:
from google.api_core import timeout
import datetime

print("Timeout and Deadline Management:")

# A timeout object is a decorator: the wrapped callable receives the computed value as its
# `timeout` keyword argument on every call.
seen = {}
constant = timeout.ConstantTimeout(5.0)(lambda **kw: seen.update(kw))
constant()
print(f"ConstantTimeout(5.0) passed timeout={seen['timeout']}")
assert seen["timeout"] == 5.0, seen

# ExponentialTimeout grows by `multiplier` on each call, capped at `maximum`.
exponential_values = []
exponential = timeout.ExponentialTimeout(initial=1.0, maximum=6.0, multiplier=2.0)(
    lambda **kw: exponential_values.append(kw["timeout"]))
for _ in range(5):
    exponential()
print(f"ExponentialTimeout(initial=1, maximum=6, multiplier=2) passed {exponential_values}")
assert exponential_values == [1.0, 2.0, 4.0, 6.0, 6.0], exponential_values

# TimeToDeadlineTimeout passes the time left until the overall deadline. A fixed clock keeps
# the numbers exact without sleeping.
now = [datetime.datetime(2024, 1, 1, tzinfo=datetime.timezone.utc)]
remaining = []
to_deadline = timeout.TimeToDeadlineTimeout(timeout=30.0, clock=lambda: now[0])(
    lambda **kw: remaining.append(kw["timeout"]))
to_deadline()
now[0] += datetime.timedelta(seconds=12)
to_deadline()
print(f"TimeToDeadlineTimeout(30) passed {remaining} at t=0s and t=12s")
assert remaining == [30.0, 18.0], remaining

print(f"\nTimeout management examples completed!")